# 本実験前の測定妥当性パイロット

3概念ペア、216入力。測定位置（文脈末尾／回答直前）、Probeの追加説明文24件/ペアでの評価、手がかり単独での距離と出力の対応を確認する。

GPUをT4に設定して「すべて実行」。固定した設計はリポジトリの1103_validity/PLAN.mdを参照。独立単位は3ペアであり、仮説の確証的検定ではない。

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install "transformers==4.57.6" "scikit-learn>=1.3,<2" "scipy>=1.11" "pandas>=2.1" "numpy>=1.26"


In [ ]:
from pathlib import Path
import subprocess,sys
REPO=Path('/content/GraduationThesis-validity')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/ryouy/GraduationThesis.git',str(REPO)],check=True)
print('SOURCE:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
subprocess.run([sys.executable,'-m','unittest','discover','-s','1103_validity','-p','test_*.py','-v'],cwd=REPO,check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), 'T4 GPUを選択してください'
OUT=Path('/content/validity-pilot-20261103')
OUT.mkdir(exist_ok=True)
with (OUT/'execution.log').open('a') as log:
    process=subprocess.Popen([sys.executable,'-u','1103_validity/run_pilot.py','--out',str(OUT)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:
        log.write(line);log.flush();print(line,end='',flush=True)
    code=process.wait()
assert code==0, 'execution.logを確認してください'


In [ ]:
import pandas as pd,json
manifest=json.loads((OUT/'manifest.json').read_text())
assert manifest['status']=='COMPLETED'
print('COMPLETED',manifest['contexts'],'contexts')
print('PROBE VALIDATION')
print(pd.read_csv(OUT/'probe_validation.csv').to_string(index=False))
print('DISTANCE AND OUTPUT')
print(pd.read_csv(OUT/'distance_correlations.csv').query('metric=="candidate_margin"').to_string(index=False))
print('POSITION EFFECTS')
print(pd.read_csv(OUT/'position_effects.csv').query('layer==23').to_string(index=False))


In [ ]:
import json,zipfile
archive=OUT.parent/'validity-pilot-results-20261103.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in sorted(OUT.iterdir()):
        if file.is_file():z.write(file,arcname=file.name)
print('Results archive:',archive,archive.stat().st_size)
# Text export is useful when Colab file downloads are unavailable.
print('VALIDITY_EXPORT_START')
print(json.dumps({p.name:p.read_text() for p in sorted(OUT.iterdir()) if p.is_file()}))
print('VALIDITY_EXPORT_END')
